# Step 06: Binary Classification (Fault vs Normal) — Canonical, V6

**Dataset**: V6 (`cooked_data_v6/features_engineered_v6.pkl`) — 2,109 events, 766 features,
ground-truth labels, corrected trigger alignment, leakage fixes applied.

**Pipeline run**: SLURM job 57924849 (`prepare_06_phase1_binary.py`, leakage-safe:
scaler/PCA fit on the train fold only via `leakage_safe_features.leakage_safe_split`).

**Why this notebook was rebuilt (2026-09-05)**: the original `06_binary_classification.ipynb`
(retired to `legacy/`) reported stale markdown numbers from a 7,427-event run that didn't match
its own live re-execution, and that live re-execution showed XGBoost at a literal 100% on every
metric -- confirmed as a real leakage bug (see `NOTEBOOK_AUDIT_2026-09-03.md` and the two
`spiral2_open_methodology_questions`/`spiral2_dataset_provenance` memory entries for the full
investigation). Root causes found and fixed: (1) `interlock_type`, a feature computed from the
exact same ALM field the label used, and (2) `StandardScaler`/PCA fit on the full dataset before
any train/test split existed. **The near-100% result below was re-verified as genuine (not
residual leakage) via direct inspection of raw signal traces** (bypassing all feature-engineering
code) showing a large, immediate post-trigger transient for every fault category, ruled out
against acquisition-config and year confounds -- see the memory entries for the full evidence
chain. This notebook loads the actual SLURM job's saved results; it does not retrain models.

In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_recall_fscore_support
)
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))
from utilities.reporting.manifest import save_manifest

RESULTS_FILE = Path('/sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6/step_06_phase1_v6/binary_classification.pkl')
SLURM_JOB = '57924849'

with open(RESULTS_FILE, 'rb') as f:
    results = pickle.load(f)

print(f"Loaded: {RESULTS_FILE}")
print(f"Available models: {[k for k in results.keys() if k != 'data_splits']}")

Loaded: /sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6/step_06_phase1_v6/binary_classification.pkl
Available models: ['logistic_regression', 'random_forest', 'xgboost', 'svm']


In [2]:
splits = results['data_splits']
y_test = splits['y_test']
n_total = len(splits['y_train']) + len(y_test)

print("="*70)
print("DATASET SUMMARY (V6, leakage-safe)")
print("="*70)
print(f"Total events: {n_total}  (train {len(splits['y_train'])} / test {len(y_test)})")
print(f"Test set — Normal: {(y_test==0).sum()}, Fault: {(y_test==1).sum()}")
print(f"Features after train-fold-fit scaling: {splits['X_train'].shape[1]}")
print(f"Scaler fit on TRAIN fold only (leakage-safe): {splits['scaler'].mean_.shape[0]} dims")

DATASET SUMMARY (V6, leakage-safe)
Total events: 2109  (train 1476 / test 633)
Test set — Normal: 320, Fault: 313
Features after train-fold-fit scaling: 766
Scaler fit on TRAIN fold only (leakage-safe): 766 dims


In [3]:
models = ['logistic_regression', 'random_forest', 'xgboost', 'svm']
model_names = ['Logistic Regression', 'Random Forest', 'XGBoost', 'SVM (RBF)']

rows = []
for key, name in zip(models, model_names):
    y_pred = results[key]['predictions']
    y_prob = results[key]['probabilities']
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    auc = results[key]['roc_auc']
    rows.append({'Method': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC': auc})

df_performance = pd.DataFrame(rows)
df_performance

,Method,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.996840,0.996805,0.996805,0.996805,0.999870
1,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000
2,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000
3,SVM (RBF),0.971564,0.945619,1.000000,0.972050,0.999271


## Interpretation

Literal 100% for LogReg/RF/XGBoost, ~97% for SVM. This pattern (three model families landing on
an exact boundary, a fourth landing close-but-not-exact) is what genuine, dramatic class
separability looks like once the trigger-alignment bug (V4) is fixed — not a leakage signature,
which was ruled out by the checks summarized in the module docstring above. Physically: any of the
7 fault categories here is a hardware safety interlock (quench, RF safety threshold, vacuum
threshold, etc.) that produces a large, near-immediate RF power/phase transient right at the
trigger; genuinely fault-free files show no such transient. Caveat: this reflects detection of
already-tripped, large-magnitude faults, not the harder problem of predicting them *before* they
happen (that's step 05, precursor detection, a separate and much less separable task).

In [4]:
metrics = {}
for _, row in df_performance.iterrows():
    slug = row['Method'].lower().replace(' ', '_').replace('(', '').replace(')', '')
    metrics[f'{slug}_accuracy'] = {'value': float(row['Accuracy']), 'fmt': '.1%', 'label': f"{row['Method']} accuracy"}
    metrics[f'{slug}_roc_auc'] = {'value': float(row['ROC_AUC']), 'fmt': '.4f', 'label': f"{row['Method']} ROC AUC"}
    metrics[f'{slug}_precision'] = {'value': float(row['Precision']), 'fmt': '.1%', 'label': f"{row['Method']} precision"}
    metrics[f'{slug}_recall'] = {'value': float(row['Recall']), 'fmt': '.1%', 'label': f"{row['Method']} recall"}
    metrics[f'{slug}_f1'] = {'value': float(row['F1']), 'fmt': '.3f', 'label': f"{row['Method']} F1"}
metrics['n_events_total'] = {'value': int(n_total), 'fmt': None, 'label': 'Total events (V6)'}
metrics['n_events_test'] = {'value': int(len(y_test)), 'fmt': None, 'label': 'Test-set events'}
metrics['n_features'] = {'value': int(splits['X_train'].shape[1]), 'fmt': None, 'label': 'Features used'}

manifest = save_manifest(
    phase='06_binary_classification',
    metrics=metrics,
    pipeline_run={
        'dataset_version': 'V6',
        'dataset_path': '/sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6/features_engineered_v6.pkl',
        'slurm_job': SLURM_JOB,
        'script': 'pipeline/00_scripts/prepare_06_phase1_binary.py',
    },
    meta={
        'leakage_investigation': 'See spiral2_open_methodology_questions memory item 5 and '
                                  'NOTEBOOK_AUDIT_2026-09-03.md for the full root-cause chain '
                                  'and the raw-signal verification that this result is genuine.',
    },
)
manifest

{'phase': '06_binary_classification',
 'generated_at': '2026-09-05T09:40:57.671642+00:00',
 'pipeline_run': {'dataset_version': 'V6',
  'dataset_path': '/sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6/features_engineered_v6.pkl',
  'slurm_job': '57924849',
  'script': 'pipeline/00_scripts/prepare_06_phase1_binary.py'},
 'metrics': {'logistic_regression_accuracy': {'value': 0.9968404423380727,
   'fmt': '.1%',
   'label': 'Logistic Regression accuracy',
   'display': '99.7%'},
  'logistic_regression_roc_auc': {'value': 0.9998702076677316,
   'fmt': '.4f',
   'label': 'Logistic Regression ROC AUC',
   'display': '0.9999'},
  'logistic_regression_precision': {'value': 0.9968051118210862,
   'fmt': '.1%',
   'label': 'Logistic Regression precision',
   'display': '99.7%'},
  'logistic_regression_recall': {'value': 0.9968051118210862,
   'fmt': '.1%',
   'label': 'Logistic Regression recall',
   'display': '99.7%'},
  'logistic_regression_f1': {'value': 0.9968051118210862,
   'fmt': '.3